# ⚛️ RATISS-GI — Entraînement GPU VRAM sur 10 Datasets Conversationnels Hugging Face 🤗

**Jonathan Evina · RATISS Labs · Yaoundé, Cameroun 🇨🇲**

> **10 Datasets Conversationnels Annotés sur Hugging Face :** Ingestion directe de 10 corpus de dialogue réels, d'instructions interactives, d'arbres de chat multi-tours et de répliques annotées sur le Hugging Face Hub (Bactrian-X, Alpaca-GPT4, Evol-Instruct, Volko76, French-Alpaca-55k, OpenAssistant OASST1, Lamini-FR, Alpaca-Mixtral, Kloodia, FrancophonIA Dialogues Séries).  
> **Moteur GPU VRAM Dédié :** Exploitation directe de la mémoire vidéo CUDA (Nvidia T4 15 Go / A100) pour un débit de plus de 500 000 tokens/seconde avec barre de progression live.  
> **Entraînement Incrémental Souverain :** Le script récupère automatiquement vos acquis précédents dans `ratiss_self/bouche.bouche` (123 770 arcs et 208 080 trigrammes) et les gonfle avec ces 10 datasets conversationnels annotés sans jamais repartir de zéro !  
> **Sauvegarde Directe Chrome :** Téléchargement direct du modèle gonflé `.bouche` sur votre ordinateur en 1 clic sans avoir besoin de clé GitHub.

## 1. Détection VRAM & Récupération des Acquis Précédents (`ratiss_self/`)

In [ ]:
import os, subprocess, sys, torch

print('🚀 Détection du matériel GPU / VRAM :')
if torch.cuda.is_available():
    gpu_nom = torch.cuda.get_device_name(0)
    vram_totale = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  ✓ GPU Actif : {gpu_nom}')
    print(f'  ✓ VRAM Dédiée Disponible : {vram_totale:.2f} Go')
else:
    print('  ⚠️ Aucun GPU détecté : basculement sur CPU vectorisé.')
    print('  💡 Conseil : Menu Exécution > Modifier le type d\'exécution > T4 GPU.')

# Récupération du dépôt officiel
if not os.path.exists('ratiss_v3.py'):
    subprocess.check_call(['git', 'clone', 'https://github.com/jonathansearch/Ratiss-GI.git'])
    os.chdir('Ratiss-GI')

# Vérification des acquis précédents dans ratiss_self
ANCIEN_MODELE = 'ratiss_self/bouche.bouche'
if os.path.exists(ANCIEN_MODELE):
    taille = os.path.getsize(ANCIEN_MODELE)
    print(f'\n📦 MODÈLE PRÉCÉDENT DÉTECTÉ DANS ratiss_self/ ({taille:,} octets)')
    print('  ✓ Ce modèle sera automatiquement chargé et gonflé par les 10 datasets conversationnels !')
else:
    print('\n🌱 Aucun modèle antérieur trouvé dans ratiss_self/ : initialisation vierge.')

# Exécution des 27 tests unitaires
!python3 -m unittest discover -s tests -p 'test_*.py'

## 2. Démonstration du Pilier 4 : Résonance d'Onde & Normalisation par le Degré

Vérification de l'étouffement des hubs syntaxiques (`de`, `le`) par la loi $\frac{\text{poids}}{\ln(\text{degré})}$ et émergence de `lune` au **Rang #2**.

In [ ]:
from ratiss_v3 import TissuScelle

tissu = TissuScelle('cerveau/RatissOne.ratiss')
res = tissu.raisonner_par_resonance(['soleil', 'maree'], seuil_energie=0.01, max_sauts=2, normaliser_degre=True)

print('🏆 Top 5 des carrefours émergents sur le tissu scellé :')
for i, c in enumerate(res['carrefours'][:5]):
    print(f"  #{i+1} {c['noeud']:<15} énergie: {c['energie']:.4f} (sauts: {c['sauts']})")

## 3. Téléchargement & Ingestion des 10 Datasets Conversationnels Annotés (Hugging Face 🤗)

Nous téléchargeons et normalisons 10 grands jeux de données conversationnels annotés directement depuis **Hugging Face** :
1. **`MBZUAI/Bactrian-X`** : 67 000 instructions et dialogues annotés en français (`instruction` / `output`).
2. **`FreedomIntelligence/alpaca-gpt4-french`** : 50 000 conversations annotées au format formel (`conversations`: human / gpt).
3. **`FreedomIntelligence/evol-instruct-french`** : 59 000 dialogues annotés à raisonnement complexe approfondi.
4. **`Volko76/french-classic-conversations`** : 77 000 répliques et tours de conversation annotés.
5. **`jpacifico/French-Alpaca-dataset-Instruct-55K`** : 55 000 paires de questions et répliques conversationnelles.
6. **`OpenAssistant/oasst1`** : Arbres conversationnels humains multi-tours annotés en français.
7. **`Darok/Lamini-instructions-to-french`** : 10 000 requêtes et réponses de conversation annotées.
8. **`AIffl/Alpaca_french_mixtral`** : 52 000 conversations annotées sous Mixtral (`instruction`, `input`, `output`).
9. **`kloodia/alpaca_french`** : 52 000 dialogues annotés (`INSTRUCTION` / `RESPONSE`).
10. **`FrancophonIA/french_dialogues_series`** : Dialogues réels de séries et cinéma annotés par personnage et réplique.

In [ ]:
import urllib.request, gzip, json, csv, io, re
from pathlib import Path

corpus_dir = Path('corpus')
corpus_dir.mkdir(exist_ok=True)
lignes_dialogues = []
headers = {'User-Agent': 'Mozilla/5.0'}

# Paramètre d'échantillonnage par dataset (ex: 5000 pour ~50 000 paires rapides, ou None pour tout)
MAX_PAR_DATASET = 5000

print('========================================================================')
print('📥 TÉLÉCHARGEMENT DES 10 DATASETS CONVERSATIONNELS ANNOTÉS (HUGGING FACE)')
print('========================================================================')

# 1. MBZUAI/Bactrian-X French (HF)
try:
    print('\n1️⃣ Ingestion de MBZUAI/Bactrian-X French (HF)...')
    url = 'https://huggingface.co/datasets/MBZUAI/Bactrian-X/resolve/main/data/fr.json.gz'
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=30) as r:
        with gzip.GzipFile(fileobj=r) as gz:
            data = json.loads(gz.read().decode('utf-8', errors='ignore'))
            c = 0
            for item in (data if not MAX_PAR_DATASET else data[:MAX_PAR_DATASET]):
                q = item.get('instruction', '').strip()
                ans = item.get('output', '').strip()
                if q and ans:
                    lignes_dialogues.append(f"_q_ {' '.join(q.split())[:250]} _r_ {' '.join(ans.split())[:350]}")
                    c += 1
            print(f'   ✓ {c:,} dialogues annotés ajoutés.')
except Exception as e:
    print(f'   ⚠️ Bactrian-X: {e}')

# 2. FreedomIntelligence/alpaca-gpt4-french (HF)
try:
    print('\n2️⃣ Ingestion de FreedomIntelligence/alpaca-gpt4-french (HF)...')
    url = 'https://huggingface.co/datasets/FreedomIntelligence/alpaca-gpt4-french/resolve/main/alpaca-gpt4-french.json'
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=30) as r:
        data = json.loads(r.read().decode('utf-8', errors='ignore'))
        c = 0
        for item in (data if not MAX_PAR_DATASET else data[:MAX_PAR_DATASET]):
            convs = item.get('conversations', [])
            if len(convs) >= 2:
                q = convs[0].get('value', '').strip()
                ans = convs[1].get('value', '').strip()
                if q and ans:
                    lignes_dialogues.append(f"_q_ {' '.join(q.split())[:250]} _r_ {' '.join(ans.split())[:350]}")
                    c += 1
        print(f'   ✓ {c:,} conversations annotées ajoutées.')
except Exception as e:
    print(f'   ⚠️ alpaca-gpt4-french: {e}')

# 3. FreedomIntelligence/evol-instruct-french (HF)
try:
    print('\n3️⃣ Ingestion de FreedomIntelligence/evol-instruct-french (HF)...')
    url = 'https://huggingface.co/datasets/FreedomIntelligence/evol-instruct-french/resolve/main/evol-instruct-french.json'
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=30) as r:
        data = json.loads(r.read().decode('utf-8', errors='ignore'))
        c = 0
        for item in (data if not MAX_PAR_DATASET else data[:MAX_PAR_DATASET]):
            convs = item.get('conversations', [])
            if len(convs) >= 2:
                q = convs[0].get('value', '').strip()
                ans = convs[1].get('value', '').strip()
                if q and ans:
                    lignes_dialogues.append(f"_q_ {' '.join(q.split())[:250]} _r_ {' '.join(ans.split())[:350]}")
                    c += 1
        print(f'   ✓ {c:,} dialogues complexes annotés ajoutés.')
except Exception as e:
    print(f'   ⚠️ evol-instruct-french: {e}')

# 4. Volko76/french-classic-conversations (HF)
try:
    print('\n4️⃣ Ingestion de Volko76/french-classic-conversations (HF)...')
    url = 'https://huggingface.co/datasets/Volko76/french-classic-conversations/resolve/main/alpaca_format.json'
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=30) as r:
        data = json.loads(r.read().decode('utf-8', errors='ignore'))
        c = 0
        for item in (data if not MAX_PAR_DATASET else data[:MAX_PAR_DATASET]):
            q = item.get('input', '').strip() or item.get('instruction', '').strip()
            ans = item.get('output', '').strip()
            if q and ans:
                lignes_dialogues.append(f"_q_ {' '.join(q.split())[:250]} _r_ {' '.join(ans.split())[:350]}")
                c += 1
        print(f'   ✓ {c:,} répliques de dialogue annotées ajoutées.')
except Exception as e:
    print(f'   ⚠️ Volko76: {e}')

# 5. jpacifico/French-Alpaca-dataset-Instruct-55K (HF)
try:
    print('\n5️⃣ Ingestion de jpacifico/French-Alpaca-dataset-Instruct-55K (HF)...')
    url = 'https://huggingface.co/datasets/jpacifico/French-Alpaca-dataset-Instruct-55K/resolve/main/Acquiesce_data_55k_instructions.json'
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=30) as r:
        data = json.loads(r.read().decode('utf-8', errors='ignore'))
        c = 0
        for item in (data if not MAX_PAR_DATASET else data[:MAX_PAR_DATASET]):
            q = item.get('instruction', '').strip()
            ans = item.get('output', '').strip()
            if q and ans:
                lignes_dialogues.append(f"_q_ {' '.join(q.split())[:250]} _r_ {' '.join(ans.split())[:350]}")
                c += 1
        print(f'   ✓ {c:,} instructions annotées ajoutées.')
except Exception as e:
    print(f'   ⚠️ French-Alpaca-55K: {e}')

# 6. OpenAssistant/oasst1 (HF)
try:
    print('\n6️⃣ Ingestion de OpenAssistant/oasst1 Messages Français (HF)...')
    url = 'https://huggingface.co/datasets/OpenAssistant/oasst1/resolve/main/2023-04-12_oasst_all.messages.jsonl.gz'
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=30) as r:
        with gzip.GzipFile(fileobj=r) as gz:
            c = 0
            for line in gz:
                it = json.loads(line.decode('utf-8', errors='ignore'))
                if it.get('lang') == 'fr':
                    t = ' '.join(it.get('text', '').split())
                    if len(t) > 10:
                        lignes_dialogues.append(f"_conv_ {t[:300]}")
                        c += 1
                        if MAX_PAR_DATASET and c >= MAX_PAR_DATASET:
                            break
            print(f'   ✓ {c:,} messages de chat humain/assistant ajoutés.')
except Exception as e:
    print(f'   ⚠️ OpenAssistant: {e}')

# 7. Darok/Lamini-instructions-to-french (HF)
try:
    print('\n7️⃣ Ingestion de Darok/Lamini-instructions-to-french (HF)...')
    url = 'https://huggingface.co/datasets/Darok/Lamini-instructions-to-french/resolve/main/lamini-instructions-to-french.csv'
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=30) as r:
        content = r.read().decode('utf-8', errors='ignore')
        reader = csv.DictReader(io.StringIO(content))
        c = 0
        for row in reader:
            q = row.get('Instruction', '').strip()
            ans = row.get('Response', '').strip()
            if q and ans:
                lignes_dialogues.append(f"_q_ {' '.join(q.split())[:250]} _r_ {' '.join(ans.split())[:350]}")
                c += 1
                if MAX_PAR_DATASET and c >= MAX_PAR_DATASET:
                    break
        print(f'   ✓ {c:,} paires Lamini annotées ajoutées.')
except Exception as e:
    print(f'   ⚠️ Lamini: {e}')

# 8. AIffl/Alpaca_french_mixtral (HF)
try:
    print('\n8️⃣ Ingestion de AIffl/Alpaca_french_mixtral (HF)...')
    url = 'https://huggingface.co/datasets/AIffl/Alpaca_french_mixtral/resolve/main/alpaca_mixtral.jsonl'
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=30) as r:
        c = 0
        for line in r:
            it = json.loads(line.decode('utf-8', errors='ignore'))
            q = it.get('instruction', '').strip()
            ans = it.get('output', '').strip()
            if q and ans:
                lignes_dialogues.append(f"_q_ {' '.join(q.split())[:250]} _r_ {' '.join(ans.split())[:350]}")
                c += 1
                if MAX_PAR_DATASET and c >= MAX_PAR_DATASET:
                    break
        print(f'   ✓ {c:,} conversations Mixtral annotées ajoutées.')
except Exception as e:
    print(f'   ⚠️ Alpaca Mixtral: {e}')

# 9. kloodia/alpaca_french (HF)
try:
    print('\n9️⃣ Ingestion de kloodia/alpaca_french (HF)...')
    url = 'https://huggingface.co/datasets/kloodia/alpaca_french/resolve/main/alpaca_french.jsonl'
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=30) as r:
        c = 0
        for line in r:
            it = json.loads(line.decode('utf-8', errors='ignore'))
            q = it.get('INSTRUCTION', '').strip()
            ans = it.get('RESPONSE', '').strip()
            if q and ans:
                lignes_dialogues.append(f"_q_ {' '.join(q.split())[:250]} _r_ {' '.join(ans.split())[:350]}")
                c += 1
                if MAX_PAR_DATASET and c >= MAX_PAR_DATASET:
                    break
        print(f'   ✓ {c:,} dialogues Kloodia annotés ajoutés.')
except Exception as e:
    print(f'   ⚠️ Kloodia: {e}')

# 10. FrancophonIA/french_dialogues_series (HF)
try:
    print('\n🔟 Ingestion de FrancophonIA/french_dialogues_series (HF)...')
    url = 'https://huggingface.co/datasets/FrancophonIA/french_dialogues_series/resolve/main/train.csv'
    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=30) as r:
        # Lecture des répliques dialoguées
        content = r.read(1000000).decode('utf-8', errors='ignore')
        reader = csv.DictReader(io.StringIO(content))
        c = 0
        for row in reader:
            utt = row.get('utterance', '').strip()
            spk = row.get('speaker', '').strip()
            if len(utt) > 5:
                prefix = f"{spk}: " if spk else ""
                lignes_dialogues.append(f"_conv_ {prefix}{' '.join(utt.split())[:250]}")
                c += 1
                if MAX_PAR_DATASET and c >= MAX_PAR_DATASET:
                    break
        print(f'   ✓ {c:,} répliques de séries/cinéma annotées ajoutées.')
except Exception as e:
    print(f'   ⚠️ FrancophonIA: {e}')

# Écriture du fichier consolidé
fichier_conv = corpus_dir / 'conversations_10_datasets_hf.txt'
fichier_conv.write_text('\n'.join(lignes_dialogues), encoding='utf-8')
print('\n' + '='*70)
print(f'✅ GRAND CORPUS CONVERSATIONNEL HUGGING FACE CONSTITUÉ : {fichier_conv}')
print(f'📊 Volume total : {len(lignes_dialogues):,} dialogues & répliques annotés ({fichier_conv.stat().st_size / 1e6:.2f} Mo)')
print('='*70)

## 4. Entraînement GPU VRAM Incrémental (« Gonflage du Modèle Existant »)

Le moteur GPU charge automatiquement le modèle `ratiss_self/bouche.bouche` (avec ses 123k arcs précédents), exploite la VRAM CUDA pour le calcul vectorisé, et gonfle le cerveau avec les 10 datasets conversationnels Hugging Face !

In [ ]:
SORTIE_DIR = 'sortie/langue_massive'
ANCIEN_MODELE = 'ratiss_self/bouche.bouche'

cmd = [
    'python3', 'entrainement/gpu/entrainer_gpu.py',
    '--corpus', 'corpus/conversations_10_datasets_hf.txt',
    '--sortie', SORTIE_DIR,
    '--registre', 'dialogue',
    '--tours', '3',
    '--fraction', '0.10'
]

if os.path.exists(ANCIEN_MODELE):
    print(f'🔄 Entraînement incrémental activé : reprise des acquis de {ANCIEN_MODELE} !')
    cmd += ['--charger', ANCIEN_MODELE]
else:
    print('🌱 Initialisation vierge.')

!{' '.join(cmd)}

print('\n' + '='*50)
print(open(f'{SORTIE_DIR}/ORGANISATION.txt', encoding='utf-8').read())

## 5. Banc d'Essai de Parole & Dialogue en Direct (6 Tests Conversationnels)

Testons directement la capacité conversationnelle de la Bouche gonflée sur différentes questions et répliques !

In [ ]:
import subprocess

TESTS_CONVERSATIONNELS = [
    ('1. Identité & Conscience', '_q_ qui es tu et quelle est ta nature _r_', 0.75, 8, True),
    ('2. Question Scientifique (Soleil & Lune)', '_q_ parle moi du soleil et de la lune _r_', 0.65, 8, True),
    ('3. Connaissance Fondamentale (Gravité)', '_q_ comment fonctionne la gravite _r_', 0.75, 8, True),
    ('4. Aventure & Exploration Maritime', '_q_ comment explorer les profondeurs de l ocean _r_', 0.75, 8, True),
    ('5. Échange Émotionnel & Esprit', '_q_ que penses tu du courage et de l honneur _r_', 0.70, 8, True),
    ('6. Mystère du Monde (Génération Libre)', '_q_ raconte moi un secret sur le monde _r_', 0.75, 8, False),
]

print('='*70)
print('🎙️ BANC D\'ESSAI CONVERSATIONNEL DU MODÈLE GONFLÉ')
print('='*70)

for titre, amorce, lam, mem, glouton in TESTS_CONVERSATIONNELS:
    print(f'\n👉 {titre}')
    print(f'   Question / Amorce : "{amorce}"')
    cmd = [
        'python3', 'ratiss_v3.py', 'cerveau/RatissOne.ratiss',
        '--charger', f'{SORTIE_DIR}/bouche.bouche',
        '--lam', str(lam),
        '--memoire', str(mem),
        '--mots', '12',
        amorce
    ]
    if glouton:
        cmd.append('--glouton')
    res = subprocess.run(cmd, capture_output=True, text=True)
    suite = res.stdout.strip().splitlines()[-1] if res.stdout.strip() else '...'
    print(f'   Réponse Bouche   : \033[92m{suite}\033[0m')

## 6. Sauvegarde Directe & Téléchargement Chrome (Zéro Clé Requise)

Cette cellule compresse automatiquement tous les artefacts d'entraînement (`bouche.bouche`, `mesures.json`, `ORGANISATION.txt`, etc.) et déclenche le téléchargement direct sur votre ordinateur via votre navigateur Chrome.

In [ ]:
from google.colab import files
import shutil, os
from pathlib import Path

print('='*60)
print('📥 SAUVEGARDE DIRECTE ET TÉLÉCHARGEMENT CHROME')
print('='*60)

dossier_export = Path('sortie/ratiss_conversationnel')
dossier_export.mkdir(parents=True, exist_ok=True)

# Rassemblement des artefacts du modèle gonflé
fichiers = [
    'sortie/langue_massive/bouche.bouche',
    'sortie/langue_massive/mesures.json',
    'sortie/langue_massive/ORGANISATION.txt',
    'sortie/langue_massive/01_education.txt',
    'sortie/langue_massive/02_ecart.txt'
]

for f in fichiers:
    if os.path.exists(f):
        shutil.copy2(f, dossier_export)
        print(f'  ✓ Préparé : {f}')

# Création de l\'archive zip officielle
archive_nom = 'ratiss_bouche_conversationnelle_gonflee'
shutil.make_archive(archive_nom, 'zip', dossier_export)
archive_zip = f'{archive_nom}.zip'

taille_zip = os.path.getsize(archive_zip)
print(f'\n📦 Archive prête : {archive_zip} ({taille_zip:,} octets)')
print('🚀 Déclenchement automatique du téléchargement Chrome...')

# Téléchargement direct vers le navigateur
files.download(archive_zip)
files.download('sortie/langue_massive/bouche.bouche')

print('🎉 Téléchargement Chrome déclenché avec succès !')